# NBA 3.1 Statistical Feature Selection (Section 4)

Removes features that cannot help the model, separately for **existing writers** and **non-writers**, using the training rows (c1-c24) of the sample list (default `all_hcps_5_train_months`: every HCP, 5 training months + c25-c26) and its weights. **Removal stages**, in order: 1 constant, 2 near-constant (one value in at least 99.5% of rows), 3 low fill (non-default value in under 1% of rows), 4 near-copies (rank correlation at least 0.95 with a stronger feature). The features left after stage 4 are the list passed to model-based selection (Section 5).

**Flags only (decided 2026-11-25, nothing removed):** 5 no signal (linear and non-linear link to NBRx both below random noise), 6 unstable (link not repeated in both halves of the doctors; checked only for features with a signal). Single-feature tests miss features that only help in combination with others, so these are left to the model-based step.

**How it fits in memory:** stages 1-3 are counts, so Spark computes them on all training rows (cell 3a); only the features left after them are loaded, 100 at a time (rows placed by a whole-number position, columns ranked in parallel), for stage 2's exact top value share, stage 4 and the flags (cell 3b). Tested on the option A sample (300 features): same stage for every feature as computing everything in memory; on the all-HCP sample a 300-feature test run succeeded (2026-11-26).

**Noise level:** NBRx is shuffled between doctors (each doctor's months kept together) and the 99th percentile of the resulting links (features that passed stages 1-3) is the bar a real feature must beat.
**Widgets:** `RUN_NAME`, `SAMPLE_NAME`, `MAX_FEATURES` (0 = all; N = test on N spread features), `WRITE_OUTPUT`.

**Iteration 3 (all HCPs).** The 10k-HCP version of this notebook is in `Iteration_2_with_10000_HCPs/`. The first full all-HCP run (job run 210413136517435, started 2026-11-26) was launched from this same notebook while it was still in the Iteration 2 folder, before the folders were reorganised; its output goes to `combined_2026_11_26_full/feature_selection/_sample_all_hcps_5_train_months/statistical/`.

In [0]:
import numpy as np, pandas as pd, json, os, time
from scipy.stats import rankdata
from pyspark.sql import functions as F

dbutils.widgets.text("RUN_NAME", "combined_2026_11_26_full")
dbutils.widgets.text("SAMPLE_NAME", "all_hcps_5_train_months")
dbutils.widgets.text("MAX_FEATURES", "0")
dbutils.widgets.dropdown("WRITE_OUTPUT", "false", ["true", "false"])
RUN_NAME     = dbutils.widgets.get("RUN_NAME").strip()
SAMPLE_NAME  = dbutils.widgets.get("SAMPLE_NAME").strip()
MAX_FEATURES = int(dbutils.widgets.get("MAX_FEATURES") or 0)
WRITE_OUTPUT = dbutils.widgets.get("WRITE_OUTPUT") == "true"

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"
COMBINED_DIR    = f"{DEV}{RUN_NAME}/hcp_combined/_sample_{SAMPLE_NAME}/"
COMBINED_PATH   = COMBINED_DIR + "kerendia_nba_3_1_hcp_selected_features_data"
DICTIONARY_PATH = COMBINED_DIR + "kerendia_nba_3_1_hcp_feature_dictionary"
TARGET_PATH     = VOL31 + "notebook_output/target_variable/target_variable"
OUT_DIR = (f"{DEV}{RUN_NAME}/feature_selection/_sample_{SAMPLE_NAME}/statistical/"
           + (f"_test_{MAX_FEATURES}_features/" if MAX_FEATURES else ""))
assert OUT_DIR.startswith(VOL31) and "kerendia_nba_3_volume" not in OUT_DIR

# Thresholds agreed in the Section 4 plan
NEAR_CONSTANT_SHARE = 0.995   # stage 2: one value in at least this weighted share of rows
MIN_FILL_SHARE      = 0.01    # stage 3: non-default value in fewer than this weighted share of rows
NEAR_COPY_RHO       = 0.95    # stage 4: |rank correlation| with a stronger kept feature
NOISE_QUANTILE      = 0.99    # stage 5: bar = this percentile of the links after shuffling NBRx between doctors
N_BINS              = 10      # stage 5: bins for the non-linear link (correlation ratio)
COPY_SAMPLE_ROWS    = 20000   # stage 4: rows used for feature-to-feature correlations
SPARK_CHUNK         = 500     # stages 1-3: features per Spark aggregation (all training rows, nothing loaded into memory)
CHUNK               = 100     # signal statistics: features loaded into memory per pass (2 million rows x 100 peaks at ~6 GB)
SEED                = 42

ID_COLS     = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT_SET", "DATASET", "PREDICTION_MONTH", "LOOKBACK_END_DATE"]
SAMPLE_COLS = ["SPLIT", "SAMPLE_GROUP", "IS_EXISTING_WRITER", "WEIGHT_GROUP", "WEIGHT_MONTH", "WEIGHT"]
GROUPS      = {"writers": True, "non_writers": False}
STAGES = {"1_constant": "same value in every row",                      # removal stages
          "2_near_constant": f"one value in at least {NEAR_CONSTANT_SHARE:.1%} of rows",
          "3_low_fill": f"non-default value in under {MIN_FILL_SHARE:.0%} of rows",
          "4_near_copy": f"rank correlation at least {NEAR_COPY_RHO} with a stronger kept feature"}
FLAGS  = {"FLAG_5_NO_SIGNAL": "linear and non-linear link to NBRx both below the noise bar",   # flags only
          "FLAG_6_UNSTABLE": "link to NBRx not repeated in both halves of the doctors (features with a signal only)"}
print(f"RUN_NAME={RUN_NAME}  SAMPLE_NAME={SAMPLE_NAME}  MAX_FEATURES={MAX_FEATURES}  WRITE_OUTPUT={WRITE_OUTPUT}")
print("Input :", COMBINED_PATH)
print("Output:", OUT_DIR)

In [0]:
combined = spark.read.format("delta").load(COMBINED_PATH)
target = (spark.read.format("delta").load(TARGET_PATH)
          .select("HCP_COHORT_ID", F.col("KERENDIA_NBRX").cast("double").alias("TARGET_NBRX")))
keys = (combined.select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "SPLIT", "IS_EXISTING_WRITER", "WEIGHT")
        .join(target, "HCP_COHORT_ID", "left").toPandas()
        .sort_values("HCP_COHORT_ID").reset_index(drop=True))
assert keys.HCP_COHORT_ID.is_unique, "duplicate rows"
assert keys.TARGET_NBRX.notna().all(), "rows without a target value"
keys["COHORT_N"] = keys.COHORT_ID.str[1:].astype(int)
# Two fixed halves of the doctors (stage 6)
keys["HALF"] = (pd.util.hash_pandas_object(keys.PROVIDER_ID, index=False).values % 2).astype(int)

dictionary = spark.read.format("delta").load(DICTIONARY_PATH).toPandas().set_index("FEATURE_NAME")
FEATURES = [c for c in combined.columns if c not in ID_COLS + SAMPLE_COLS]
assert set(FEATURES) == set(dictionary.index), "combined table and dictionary differ"
if MAX_FEATURES:
    FEATURES = FEATURES[::max(1, len(FEATURES) // MAX_FEATURES)][:MAX_FEATURES]
FILL = dictionary.IMPUTE_VALUE.reindex(FEATURES).astype(float).values   # NaN = no fill rule (always counts as filled)

def doctor_block_permutation(df, rng):
    """NBRx shuffled between doctors with the same number of rows; each doctor's months stay together."""
    d = pd.DataFrame({"pos": np.arange(len(df)), "P": df.PROVIDER_ID.values, "k": df.COHORT_N.values, "y": df.TARGET_NBRX.values})
    d["n"] = d.groupby("P").P.transform("size")
    d = d.sort_values(["n", "P", "k"])
    out = np.empty(len(d))
    for n, sub in d.groupby("n", sort=False):
        y = sub.y.values.reshape(-1, n)
        out[sub.pos.values] = y[rng.permutation(len(y))].ravel()
    return out

rng = np.random.default_rng(SEED)
IDX, SUB, PERM, group_rows = {}, {}, {}, []
for g, flag in GROUPS.items():
    m = keys.IS_EXISTING_WRITER.values == flag
    tr = np.where(m & (keys.SPLIT.values == "train"))[0]
    te = np.where(m & (keys.SPLIT.values == "test"))[0]
    IDX[g] = {"train": tr, "test": te}
    SUB[g] = np.sort(rng.choice(len(tr), size=min(COPY_SAMPLE_ROWS, len(tr)), replace=False))
    PERM[g] = doctor_block_permutation(keys.iloc[tr], rng)
    k = keys.iloc[tr]
    assert np.isclose(PERM[g].sum(), k.TARGET_NBRX.sum()), "shuffle changed the NBRx total"
    group_rows.append((g, len(tr), k.PROVIDER_ID.nunique(), float(k.WEIGHT.sum()), float(k.TARGET_NBRX.sum()),
                       int((k.TARGET_NBRX > 0).sum()), len(te), int((keys.TARGET_NBRX.values[te] > 0).sum())))
group_summary = pd.DataFrame(group_rows, columns=["group", "train_rows", "train_doctors", "train_weighted_rows", "train_nbrx",
                                                  "train_rows_with_nbrx", "test_rows", "test_rows_with_nbrx"])
print(f"{len(FEATURES):,} candidate features; {len(keys):,} rows")
display(group_summary)

In [0]:
# Stages 1-3 only need counts, so Spark computes them on every training row without loading the data into memory.
# CONSTANT = min equals max; FILL_SHARE = weighted share of rows not equal to the fill value (compared as float, the
# stored type). TOP_VALUE_SHARE is exact here whenever the fill value is the most common value (share >= 50%), which
# covers every feature removed at stages 1-3, so the removal stages are the same as computing everything in memory.
# The remaining features get their exact TOP_VALUE_SHARE in the next cell.
FLAG_TO_GROUP = {flag: g for g, flag in GROUPS.items()}
FLOAT_COLS = {s.name for s in combined.schema.fields if s.dataType.typeName() == "float"}
FILL32 = {f: (None if pd.isna(v) else float(np.float32(v))) for f, v in zip(FEATURES, FILL)}   # pandas check (float32)
FILL_SPARK = {f: (None if pd.isna(v) else (F.lit(float(v)).cast("float") if f in FLOAT_COLS else F.lit(float(v))))
              for f, v in zip(FEATURES, FILL)}   # fill value in the column's stored type
cb_train = combined.filter("SPLIT = 'train'")
spark_parts = {g: [] for g in GROUPS}
t_start = time.time()
for start in range(0, len(FEATURES), SPARK_CHUNK):
    cols = FEATURES[start:start + SPARK_CHUNK]
    aggs = [F.sum("WEIGHT").alias("_W")]
    for i, c in enumerate(cols):
        x = F.col(f"`{c}`")
        filled = F.lit(True) if FILL_SPARK[c] is None else (x != FILL_SPARK[c])
        aggs += [F.min(x).alias(f"mn{i}"), F.max(x).alias(f"mx{i}"),
                 F.sum(F.when(filled, F.col("WEIGHT")).otherwise(F.lit(0.0))).alias(f"fw{i}")]
    for r in cb_train.groupBy("IS_EXISTING_WRITER").agg(*aggs).collect():
        spark_parts[FLAG_TO_GROUP[r["IS_EXISTING_WRITER"]]].append(pd.DataFrame({
            "FEATURE": cols,
            "CONSTANT": [r[f"mn{i}"] == r[f"mx{i}"] for i in range(len(cols))],
            "FILL_SHARE": [r[f"fw{i}"] / r["_W"] for i in range(len(cols))]}))
    print(f"stages 1-3: features {start + len(cols):,}/{len(FEATURES):,}  ({(time.time() - t_start) / 60:.1f} min)")

PRE = {}
for g in GROUPS:
    p = pd.concat(spark_parts[g], ignore_index=True).set_index("FEATURE").reindex(FEATURES)
    assert p.notna().all().all(), f"{g}: Spark statistics missing for some features"
    default_share = 1 - p.FILL_SHARE
    p["TOP_VALUE_SHARE"] = np.where(p.CONSTANT, 1.0, np.where(default_share >= 0.5, default_share, np.nan))
    p["PASSES_1_3"] = (~p.CONSTANT) & ~(p.TOP_VALUE_SHARE >= NEAR_CONSTANT_SHARE) & (p.FILL_SHARE >= MIN_FILL_SHARE)
    PRE[g] = p
    print(f"{g}: {int(p.PASSES_1_3.sum()):,} of {len(p):,} features left after stages 1-3 (stage 2 finished in the next cell)")
SURVIVORS = [f for f in FEATURES if any(PRE[g].PASSES_1_3[f] for g in GROUPS)]
print(f"{len(SURVIVORS):,} features go to the signal statistics")

In [0]:
def wcorr(R, v, w):
    """Weighted Pearson correlation of each column of R with v (on ranks = weighted Spearman)."""
    W = w.sum()
    Rc = R - (w @ R) / W
    vc = v - (w @ v) / W
    with np.errstate(invalid="ignore", divide="ignore"):
        r = ((w * vc) @ Rc) / np.sqrt((w @ (Rc ** 2)) * (w @ (vc ** 2)))
    return np.nan_to_num(r)

def weta(R, y, w):
    """Weighted correlation ratio: how much NBRx differs between 10 rank bins of the feature (catches non-linear links)."""
    n, p = R.shape
    bins = np.minimum(((R - 1) / n * N_BINS).astype(int), N_BINS - 1)
    W = w.sum(); ybar = (w @ y) / W
    tot = w @ (y - ybar) ** 2
    out = np.zeros(p)
    for j in range(p):
        wb = np.bincount(bins[:, j], weights=w, minlength=N_BINS)
        yb = np.bincount(bins[:, j], weights=w * y, minlength=N_BINS)
        ok = wb > 0
        out[j] = (wb[ok] * (yb[ok] / wb[ok] - ybar) ** 2).sum()
    return np.sqrt(out / tot) if tot > 0 else out

def top_share(X, w):
    """Weighted share of rows holding each column's most common value."""
    out = np.empty(X.shape[1])
    for j in range(X.shape[1]):
        _, inv = np.unique(X[:, j], return_inverse=True)
        out[j] = np.bincount(inv, weights=w).max() / w.sum()
    return out

stats_parts = {g: [] for g in GROUPS}
copy_ranks  = {g: [] for g in GROUPS}
copy_names  = {g: [] for g in GROUPS}
W_ALL, Y_ALL, H_ALL = keys.WEIGHT.values, keys.TARGET_NBRX.values, keys.HALF.values
# Rows are loaded with a whole-number position instead of the text key (much faster to send and place than
# matching 2 million text keys), and columns are ranked in parallel on all driver cores (same ranks).
import os
from concurrent.futures import ThreadPoolExecutor
POS_MAP = spark.createDataFrame(pd.DataFrame({"HCP_COHORT_ID": keys.HCP_COHORT_ID.values,
                                              "_POS": np.arange(len(keys), dtype=np.int32)}))
N_CORES = os.cpu_count() or 1

def load_columns(cs):
    pdf = combined.join(POS_MAP, "HCP_COHORT_ID").select("_POS", *[F.col(f"`{c}`") for c in cs]).toPandas()
    assert len(pdf) == len(keys) and pdf["_POS"].nunique() == len(keys), "rows missing or repeated in the combined table"
    X = np.empty((len(keys), len(cs)), dtype=np.float32)
    X[pdf["_POS"].to_numpy()] = pdf[cs].to_numpy(dtype=np.float32)
    return X

def prank(X):
    """rankdata(X, axis=0), one column per thread."""
    with ThreadPoolExecutor(N_CORES) as ex:
        return np.column_stack(list(ex.map(lambda j: rankdata(X[:, j]), range(X.shape[1]))))

t_start = time.time()
for start in range(0, len(SURVIVORS), CHUNK):
    cols = SURVIVORS[start:start + CHUNK]
    X_all = load_columns(cols)
    assert not np.isnan(X_all).any(), "empty values or unmatched rows in the combined table"
    for g in GROUPS:
        jj = np.array([j for j, c in enumerate(cols) if PRE[g].PASSES_1_3[c]], dtype=int)   # this group's survivors
        if len(jj) == 0:
            continue
        gc = [cols[j] for j in jj]
        tr, te = IDX[g]["train"], IDX[g]["test"]
        X, w, y, half = X_all[np.ix_(tr, jj)], W_ALL[tr], Y_ALL[tr], H_ALL[tr]
        # Built-in check against Spark (cell 3a): same fill share, and same top value share where Spark had it
        fill = np.array([np.nan if FILL32[c] is None else FILL32[c] for c in gc], dtype=np.float32)
        fill_share = (w @ (X != fill).astype(np.float32)) / w.sum()
        assert np.allclose(fill_share, PRE[g].FILL_SHARE[gc].values, rtol=0, atol=1e-6), f"{g}: fill share differs from Spark"
        top = top_share(X, w)
        known = PRE[g].TOP_VALUE_SHARE[gc].notna().values
        assert np.allclose(top[known], PRE[g].TOP_VALUE_SHARE[gc].values[known], rtol=0, atol=1e-6), f"{g}: top value share differs from Spark"
        R = prank(X)
        st = pd.DataFrame({"FEATURE": gc, "TOP_VALUE_SHARE_PD": top,
                           "RHO": wcorr(R, rankdata(y), w), "ETA": weta(R, y, w),
                           "RHO_NOISE": wcorr(R, rankdata(PERM[g]), w), "ETA_NOISE": weta(R, PERM[g], w)})
        for h, lab in [(0, "A"), (1, "B")]:
            m = half == h
            Rh = prank(X[m])
            st[f"RHO_HALF_{lab}"] = wcorr(Rh, rankdata(y[m]), w[m])
            st[f"ETA_HALF_{lab}"] = weta(Rh, y[m], w[m])
        st["RHO_TEST"] = wcorr(prank(X_all[np.ix_(te, jj)]), rankdata(Y_ALL[te]), W_ALL[te])
        stats_parts[g].append(st)
        keep = top < NEAR_CONSTANT_SHARE   # stage 2 for the survivors; stages 1 and 3 already passed in Spark
        if keep.any():
            copy_ranks[g].append(rankdata(X[SUB[g]][:, keep], axis=0).astype(np.float32))
            copy_names[g] += [c for c, k in zip(gc, keep) if k]
    print(f"features {start + len(cols):,}/{len(SURVIVORS):,}  ({(time.time() - t_start) / 60:.1f} min)")
print({g: len(v) for g, v in copy_names.items()}, "features reach stage 4")

In [0]:
RESULTS, THRESHOLDS, KEPT_MAX_COPY_RHO = {}, {}, {}
for g in GROUPS:
    # Spark stages 1-3 for every feature + signal statistics for the survivors (empty for features removed at 1-3)
    st = PRE[g].drop(columns="PASSES_1_3").rename_axis("FEATURE").reset_index()
    if stats_parts[g]:
        st = st.merge(pd.concat(stats_parts[g], ignore_index=True), on="FEATURE", how="left")
        st["TOP_VALUE_SHARE"] = st.pop("TOP_VALUE_SHARE_PD").fillna(st.TOP_VALUE_SHARE)
    assert st.TOP_VALUE_SHARE.notna().all(), f"{g}: top value share missing"
    st["CATEGORY"] = dictionary.CATEGORY.reindex(st.FEATURE).values
    live = st.RHO.notna()   # noise bars from the features that passed stages 1-3
    thr_rho = float(np.quantile(st.RHO_NOISE[live].abs(), NOISE_QUANTILE))
    thr_eta = float(np.quantile(st.ETA_NOISE[live], NOISE_QUANTILE))
    THRESHOLDS[g] = {"rho_noise_bar": thr_rho, "eta_noise_bar": thr_eta}
    st["SIGNAL_LINEAR"] = st.RHO.abs() >= thr_rho
    st["SIGNAL_NONLINEAR"] = st.ETA >= thr_eta
    st["SIGNAL_SCORE"] = np.maximum(st.RHO.abs() / thr_rho, st.ETA / thr_eta)   # 1 = at the noise bar
    st["REMOVED_AT"] = None

    def drop(mask, stage):
        m = np.asarray(mask) & st.REMOVED_AT.isna().values
        st.loc[m, "REMOVED_AT"] = stage

    drop(st.CONSTANT, "1_constant")
    drop(st.TOP_VALUE_SHARE >= NEAR_CONSTANT_SHARE, "2_near_constant")
    drop(st.FILL_SHARE < MIN_FILL_SHARE, "3_low_fill")

    # Stage 4: strongest feature first; a feature is dropped if it is a near-copy of one already kept
    names = copy_names[g]
    assert set(names) == set(st.FEATURE[st.REMOVED_AT.isna()]), "stage 4 candidates do not match stages 1-3"
    Z = np.hstack(copy_ranks[g])
    sd = Z.std(axis=0)
    Z = np.where(sd > 0, (Z - Z.mean(axis=0)) / np.where(sd > 0, sd, 1), 0).astype(np.float32)
    C = (Z.T @ Z) / len(Z)
    del Z
    sc = st.set_index("FEATURE").loc[names]
    order = np.lexsort((-sc.FILL_SHARE.values, -sc.SIGNAL_SCORE.values))
    kept, copy_of = [], {}
    for i in order:
        if kept:
            c = np.abs(C[i, kept]); j = int(c.argmax())
            if c[j] >= NEAR_COPY_RHO:
                copy_of[names[i]] = (names[kept[j]], float(C[i, kept[j]]))
                continue
        kept.append(i)
    st["COPY_OF"] = st.FEATURE.map(lambda f: copy_of.get(f, (None, None))[0])
    st["RHO_WITH_COPY"] = st.FEATURE.map(lambda f: copy_of.get(f, (None, None))[1])
    drop(st.FEATURE.isin(list(copy_of)), "4_near_copy")

    # Stages 5 and 6 are flags only: the features stay in the list for model-based selection
    st["SELECTED"] = st.REMOVED_AT.isna()
    st["REASON"] = st.REMOVED_AT.map(STAGES)
    no_signal = ~st.SIGNAL_LINEAR & ~st.SIGNAL_NONLINEAR
    unstable = np.where(st.SIGNAL_LINEAR, np.sign(st.RHO_HALF_A) != np.sign(st.RHO_HALF_B),
                        np.minimum(st.ETA_HALF_A, st.ETA_HALF_B) < thr_eta)
    st["FLAG_5_NO_SIGNAL"] = st.SELECTED & no_signal
    st["FLAG_6_UNSTABLE"] = st.SELECTED & ~no_signal & unstable
    st["PASSES_ALL_SIX"] = st.SELECTED & ~st.FLAG_5_NO_SIGNAL & ~st.FLAG_6_UNSTABLE

    sel_idx = [names.index(f) for f in st.FEATURE[st.SELECTED]]
    Cs = np.abs(C[np.ix_(sel_idx, sel_idx)]); np.fill_diagonal(Cs, 0)
    KEPT_MAX_COPY_RHO[g] = float(Cs.max()) if len(sel_idx) > 1 else 0.0
    del C, Cs
    RESULTS[g] = st
    print(f"{g}: noise bars rho={thr_rho:.4f} eta={thr_eta:.4f}; kept after stage 4: {int(st.SELECTED.sum()):,} of {len(st):,}; "
          f"flagged no signal {int(st.FLAG_5_NO_SIGNAL.sum()):,}, unstable {int(st.FLAG_6_UNSTABLE.sum()):,}; no flag {int(st.PASSES_ALL_SIX.sum()):,}")

# Funnel: features left after each removal stage, then the flags among the kept features
funnel = []
for g, st in RESULTS.items():
    left = len(st)
    funnel.append((g, "0_start", "all candidate features", 0, left, None))
    for s, d in STAGES.items():
        n = int((st.REMOVED_AT == s).sum()); left -= n
        funnel.append((g, s, d, n, left, None))
    for f, d in FLAGS.items():
        funnel.append((g, f, d, 0, left, int(st[f].sum())))
    funnel.append((g, "no_flag", "kept and not flagged", 0, left, int(st.PASSES_ALL_SIX.sum())))
funnel = pd.DataFrame(funnel, columns=["group", "stage", "rule", "removed", "features_left", "flagged"])
display(funnel)

category_summary = (pd.concat([st.assign(group=g) for g, st in RESULTS.items()])
                    .groupby(["group", "CATEGORY"]).agg(features=("FEATURE", "size"), kept_after_stage_4=("SELECTED", "sum"),
                                                        flag_no_signal=("FLAG_5_NO_SIGNAL", "sum"), flag_unstable=("FLAG_6_UNSTABLE", "sum"),
                                                        no_flag=("PASSES_ALL_SIX", "sum")).reset_index())
display(category_summary)

In [0]:
for g, st in RESULTS.items():
    sel = st[st.SELECTED]
    assert len(st) == len(FEATURES) and st.FEATURE.is_unique
    assert (st.REMOVED_AT.isna() == st.SELECTED).all()
    assert int(funnel[funnel.group == g].removed.sum()) + len(sel) == len(st), "stage counts do not add up"
    assert not sel.CONSTANT.any() and (sel.TOP_VALUE_SHARE < NEAR_CONSTANT_SHARE).all() and (sel.FILL_SHARE >= MIN_FILL_SHARE).all()
    assert KEPT_MAX_COPY_RHO[g] < NEAR_COPY_RHO, f"{g}: two kept features are near-copies"
    assert not (st.FLAG_5_NO_SIGNAL & st.FLAG_6_UNSTABLE).any(), "a feature has both flags"
    assert not (~st.SELECTED & (st.FLAG_5_NO_SIGNAL | st.FLAG_6_UNSTABLE)).any(), "a removed feature is flagged"
    assert (st.FLAG_5_NO_SIGNAL | st.FLAG_6_UNSTABLE | st.PASSES_ALL_SIX).sum() == len(sel)
    print(f"{g}: rules hold for all {len(sel):,} kept; largest |rho| between two kept = {KEPT_MAX_COPY_RHO[g]:.3f}")

# Independent re-computation in Spark of the weighted fill share for 3 features per group
cb_t = combined.filter("SPLIT = 'train'")
for g, flag in GROUPS.items():
    st = RESULTS[g].set_index("FEATURE")
    pick = list(st.index[st.SELECTED][:2]) + list(st.index[st.REMOVED_AT == "3_low_fill"][:1])
    sub = cb_t.filter(F.col("IS_EXISTING_WRITER") == flag)
    for f in pick:
        fv = dictionary.IMPUTE_VALUE[f]
        cond = F.lit(True) if pd.isna(fv) else (F.col(f"`{f}`") != F.lit(float(fv)))
        r = sub.agg((F.sum(F.when(cond, F.col("WEIGHT")).otherwise(0)) / F.sum("WEIGHT")).alias("s")).first()["s"]
        assert abs(r - st.loc[f, "FILL_SHARE"]) < 1e-9, f"{g} {f}: Spark {r} vs notebook {st.loc[f, 'FILL_SHARE']}"
    print(f"{g}: fill share matches Spark for {pick}")

# Known case: the writer rule column must be constant (all zero) among non-writers
w12 = "TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH"
if w12 in FEATURES:
    assert RESULTS["non_writers"].set_index("FEATURE").loc[w12, "REMOVED_AT"] == "1_constant"
    print(f"{w12} is constant among non-writers, as expected")

In [0]:
if WRITE_OUTPUT:
    os.makedirs(OUT_DIR, exist_ok=True)
    for g, st in RESULTS.items():
        st.sort_values(["SELECTED", "SIGNAL_SCORE"], ascending=False).to_csv(f"{OUT_DIR}feature_stats_{g}.csv", index=False)
        # List for Section 5: every feature kept after stage 4, with the two flags
        (st[st.SELECTED].sort_values("SIGNAL_SCORE", ascending=False)
           [["FEATURE", "CATEGORY", "FLAG_5_NO_SIGNAL", "FLAG_6_UNSTABLE", "PASSES_ALL_SIX", "SIGNAL_SCORE", "RHO", "ETA",
             "RHO_TEST", "FILL_SHARE", "TOP_VALUE_SHARE"]]
           .to_csv(f"{OUT_DIR}selected_features_{g}.csv", index=False))
        (st[st.REMOVED_AT == "4_near_copy"][["FEATURE", "CATEGORY", "COPY_OF", "RHO_WITH_COPY", "SIGNAL_SCORE"]]
           .to_csv(f"{OUT_DIR}near_copies_{g}.csv", index=False))
    funnel.to_csv(OUT_DIR + "funnel.csv", index=False)
    category_summary.to_csv(OUT_DIR + "category_summary.csv", index=False)
    group_summary.to_csv(OUT_DIR + "group_summary.csv", index=False)
    with open(OUT_DIR + "run_summary.json", "w") as fh:
        json.dump({"run_name": RUN_NAME, "sample_name": SAMPLE_NAME, "input": COMBINED_PATH, "n_features": len(FEATURES),
                   "thresholds": {"near_constant_share": NEAR_CONSTANT_SHARE, "min_fill_share": MIN_FILL_SHARE,
                                  "near_copy_rho": NEAR_COPY_RHO, "noise_quantile": NOISE_QUANTILE, "n_bins": N_BINS,
                                  "copy_sample_rows": COPY_SAMPLE_ROWS, "seed": SEED},
                   "noise_bars": THRESHOLDS, "selected": {g: int(st.SELECTED.sum()) for g, st in RESULTS.items()},
                   "flagged": {g: {f: int(st[f].sum()) for f in FLAGS} for g, st in RESULTS.items()},
                   "no_flag": {g: int(st.PASSES_ALL_SIX.sum()) for g, st in RESULTS.items()},
                   "max_rho_between_selected": KEPT_MAX_COPY_RHO, "minutes": round((time.time() - t_start) / 60, 1)}, fh, indent=2)
    print("Saved to", OUT_DIR, sorted(os.listdir(OUT_DIR)))
else:
    print("WRITE_OUTPUT = false: nothing saved")